# Dogs vs Cats — Image Classification with EfficientNetV2S

Binary image classifier using transfer learning with EfficientNetV2S pre-trained on ImageNet.  
Dataset: 2,000 images (subset of [Kaggle Dogs vs. Cats](https://www.kaggle.com/c/dogs-vs-cats/data))  
Target: >99% training and validation accuracy

## 1. Setup & Imports

In [ ]:
import os
import zipfile
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf
from tensorflow.keras import Model, layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import efficientnet_v2 as effnet
from tensorflow.keras.optimizers import RMSprop
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

print("TensorFlow version:", tf.__version__)

## 2. Download & Extract Dataset

In [ ]:
!wget --no-check-certificate \
    http://web.eece.maine.edu/~zhu/DL/cats_and_dogs_filtered.zip \
    -O /tmp/cats_and_dogs_filtered.zip

In [ ]:
local_zip = '/tmp/cats_and_dogs_filtered.zip'
with zipfile.ZipFile(local_zip, 'r') as zip_ref:
    zip_ref.extractall('/tmp')

base_dir        = '/tmp/cats_and_dogs_filtered'
train_dir       = os.path.join(base_dir, 'train')
validation_dir  = os.path.join(base_dir, 'validation')

train_cats_dir      = os.path.join(train_dir, 'cats')
train_dogs_dir      = os.path.join(train_dir, 'dogs')
validation_cats_dir = os.path.join(validation_dir, 'cats')
validation_dogs_dir = os.path.join(validation_dir, 'dogs')

print("Dataset extracted successfully.")

## 3. Data Exploration

In [ ]:
# Image counts per split
dirs = {
    'Train Cats':      train_cats_dir,
    'Train Dogs':      train_dogs_dir,
    'Validation Cats': validation_cats_dir,
    'Validation Dogs': validation_dogs_dir
}

for name, path in dirs.items():
    print(f"{name}: {len(os.listdir(path))} images")

In [ ]:
# Visualize 3 random samples from each split
N_SAMPLES = 3
dir_list  = list(dirs.items())

fig, axes = plt.subplots(4, N_SAMPLES, figsize=(N_SAMPLES * 4, 16))
fig.suptitle('Sample Images per Split', fontsize=16, y=1.01)

for row, (split_name, split_path) in enumerate(dir_list):
    samples = random.sample(os.listdir(split_path), N_SAMPLES)
    for col, fname in enumerate(samples):
        axes[row, col].imshow(mpimg.imread(os.path.join(split_path, fname)))
        axes[row, col].set_title(fname, fontsize=8)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(split_name, fontsize=11, rotation=90, labelpad=10)

plt.tight_layout()
plt.savefig('results/sample_images.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Data Generators & Augmentation

In [ ]:
IMG_SIZE   = (260, 260)
BATCH_SIZE = 20

# Training: augmentation + rescale
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation: rescale only
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

validation_generator = val_datagen.flow_from_directory(
    validation_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

print("Class indices:", train_generator.class_indices)

## 5. Model — EfficientNetV2S Fine-Tuning

EfficientNetV2S pre-trained on ImageNet is used as the backbone with all layers set to trainable (full fine-tuning). A custom classification head is appended on top.

In [ ]:
# Load base model
base_model = effnet.EfficientNetV2S(
    include_top=False,
    weights='imagenet',
    input_shape=(260, 260, 3),
    include_preprocessing=False
)

# Full fine-tuning — all layers trainable
for layer in base_model.layers:
    layer.trainable = True

# Custom classification head
last_output = base_model.get_layer('top_activation').output
x = layers.Flatten()(last_output)
x = layers.Dense(1024, activation='relu')(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(1, activation='sigmoid')(x)

model = Model(base_model.input, x)

model.compile(
    loss='binary_crossentropy',
    optimizer=RMSprop(learning_rate=1e-4),
    metrics=['acc']
)

print(f"Total params: {model.count_params():,}")
print(f"Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

In [ ]:
# Model architecture diagram
tf.keras.utils.plot_model(model, show_shapes=True, dpi=80)

## 6. Training

In [ ]:
EPOCHS           = 10
STEPS_PER_EPOCH  = 100
VALIDATION_STEPS = 50

history = model.fit(
    train_generator,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=VALIDATION_STEPS,
    verbose=1
)

## 7. Training Curves

In [ ]:
acc      = history.history['acc']
val_acc  = history.history['val_acc']
loss     = history.history['loss']
val_loss = history.history['val_loss']
epochs   = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, acc,     'b-o', label='Train Accuracy')
ax1.plot(epochs, val_acc, 'r-o', label='Val Accuracy')
ax1.set_title('Training vs Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, loss,     'b-o', label='Train Loss')
ax2.plot(epochs, val_loss, 'r-o', label='Val Loss')
ax2.set_title('Training vs Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('results/training_curves.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Final Train Accuracy:      {acc[-1]:.4f}")
print(f"Final Validation Accuracy: {val_acc[-1]:.4f}")

## 8. Performance Analysis

In [ ]:
THRESHOLD = 0.6

def evaluate_split(generator, steps, split_name):
    generator.reset()
    generator.shuffle = False
    y_prob  = model.predict(generator, steps=steps, verbose=0)
    y_pred  = (y_prob.flatten() > THRESHOLD).astype(int)
    y_truth = generator.classes[:len(y_pred)]

    cm = confusion_matrix(y_truth, y_pred)
    print(f"\n{'='*40}")
    print(f"  {split_name}")
    print(f"{'='*40}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Precision : {precision_score(y_truth, y_pred):.4f}")
    print(f"Recall    : {recall_score(y_truth, y_pred):.4f}")
    print(f"F1 Score  : {f1_score(y_truth, y_pred):.4f}")
    return cm

cm_train = evaluate_split(train_generator,      steps=100, split_name='Train Set')
cm_val   = evaluate_split(validation_generator, steps=50,  split_name='Validation Set')

In [ ]:
# Visualize confusion matrices
import itertools

def plot_confusion_matrix(cm, title, ax):
    classes = ['Cat', 'Dog']
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(title, fontsize=13)
    plt.colorbar(im, ax=ax)
    tick_marks = range(len(classes))
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(classes)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(classes)
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_confusion_matrix(cm_train, 'Confusion Matrix — Train', ax1)
plot_confusion_matrix(cm_val,   'Confusion Matrix — Validation', ax2)
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()